In [ ]:
import os
import gc
import torch
from pathlib import Path

from harreman_funcs import HarremanRunner
import harreman_summary

In [ ]:
XENIUM_DATA_DIR = '/global/scratch/users/fosterangus/MetabTravLR/Data/Xenium'
TIERS = ['Tier1', 'Tier2', 'Tier3']

In [ ]:
def run_dataset(data_dir, dataset_name):
    harRunner = HarremanRunner(f'{data_dir}/{dataset_name}')
    harRunner.load_adata()
    harRunner.save_harreman_network()

    tiers = [tier for tier in TIERS if tier in harRunner.adata.obs.columns]
    if not tiers:
        raise ValueError('no tier annotations')

    for tier in tiers:
        harRunner.run_harreman(tier)

    out_path = harRunner.easy_download_path
    master, genepairs = harreman_summary.summarize_harreman_folder(out_path, sample_id=dataset_name)
    summary_dir = Path(out_path) / 'summary'
    summary_dir.mkdir(parents=True, exist_ok=True)
    master.to_csv(summary_dir / 'metabolite_summary.csv', index=False)
    genepairs.to_csv(summary_dir / 'gene_pair_summary.csv', index=False)
    harreman_summary.select_tcell_metabolites(out_path)

    # written last so a dataset is only skipped once it finished
    marker_path(data_dir, dataset_name).write_text(dataset_name)

In [ ]:
def marker_path(data_dir, dataset_name):
    return Path(f'{data_dir}/{dataset_name}/easy_download/.dataset_name')


def run_all(data_dir=XENIUM_DATA_DIR):
    for dataset_name in sorted(os.listdir(data_dir)):
        if not os.path.isdir(f'{data_dir}/{dataset_name}'):
            continue
        if marker_path(data_dir, dataset_name).is_file():
            print(f'skipping {dataset_name}, already done')
            continue

        print(f'running {dataset_name}')
        try:
            run_dataset(data_dir, dataset_name)
            print(f'finished {dataset_name}')
        except Exception as e:
            print(f'failed {dataset_name}: {type(e).__name__}: {e}')

        gc.collect()
        torch.cuda.empty_cache()

In [ ]:
run_all()